In [ ]:
# Cell 0 — 00_env_validation.ipynb
# Fail fast before any expensive compute. Run standalone or via pipeline_full.ipynb.
import sys, importlib, subprocess

REQUIRED = {
    "dolfinx":   "0.8.0",
    "gmsh":      "4.12.2",   # src/meshing/gmsh_pipeline.py, src/meshing/mesh_quality.py
    "pyvista":   "0.43.10",  # 03_fea_fenicsx.ipynb, 05_stl_export.ipynb
    "trimesh":   "4.4.0",    # 05_stl_export.ipynb
    "skimage":   "0.22.0",   # 05_stl_export.ipynb — marching cubes
    "papermill": "2.6.0",    # pipeline_full.ipynb orchestrator
}

failures = []
for pkg, expected in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        actual = getattr(mod, "__version__", "unknown")
        status = "✓" if actual == expected else f"⚠ got {actual}"
        print(f"  {pkg:<20} {expected}  {status}")
    except ImportError as e:
        failures.append(pkg)
        print(f"  {pkg:<20} MISSING — {e}")

# MPI — exercised by 03_fea_fenicsx.ipynb parallel solves
from mpi4py import MPI
print(f"\n  MPI comm size: {MPI.COMM_WORLD.Get_size()}")
print(f"  MPI rank:      {MPI.COMM_WORLD.Get_rank()}")

# OpenSCAD — runtime dep for src/geometry/openscad_runner.py
result = subprocess.run(["openscad", "--version"], capture_output=True, text=True)
print(f"  OpenSCAD:      {(result.stdout + result.stderr).strip().splitlines()[0]}")

# PyVista headless — required by 03_fea_fenicsx.ipynb and 05_stl_export.ipynb
import pyvista as pv
pv.OFF_SCREEN = True
pv.Plotter().close()
print(f"  PyVista headless render: OK")

if failures:
    raise EnvironmentError(f"Missing packages: {failures}")

print("\n✅ Environment validated — safe to proceed to 01_geometry_openscad.ipynb")